# Blackjack: Simulation and Strategy comparison 

## Deck of cards
In blackjack only the value of a card matters, so a card is represented by its value.
Number cards are worth their number, J, Q and K are worth 10, and an ace is worth 11 for now.

In [3]:
import random 
suit = [2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 10, 11]
deck = suit * 4 
random.shuffle(deck)
card = deck.pop()
print(len(deck))
print(card)

51
5


## Value of a hand
The value of a hand is the sum of its cards:

$$V = \sum_{i=1}^{n} c_i$$

If $V > 21$ and the hand contains an ace counted as 11, the ace is counted as 1 instead. This lowers $V$ by 10. We repeat this for every ace until $V \le 21$ or there are no more aces to change.

In [4]:
def hand_value(hand):
    total = sum(hand)
    aces = hand.count(11)

    while total > 21 and aces > 0:
        total -= 10
        aces -= 1

    return total

test_hands = [[10, 7], [11, 7], [11, 7, 9], [11, 11], [10, 6, 8]]

for test_hand in test_hands:
    print(hand_value(test_hand))

17
18
17
12
24


## Playing one game
The player draws cards until the hand value reaches a chosen threshold, then stops. If the player's value is over 21, the player loses (a "bust"). The dealer then draws until reaching at least 17. The higher value not over 21 wins, and equal values are a draw.

In [5]:
import random

suit = [2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 10, 11]

def hand_value(hand):
    value = sum(hand)
    aces = hand.count(11)
    while value > 21 and aces:
        value -= 10
        aces -= 1
    return value

def play_game(player_stand_at=17):
    deck = suit * 4
    random.shuffle(deck)
    player = [deck.pop(), deck.pop()]
    dealer = [deck.pop(), deck.pop()]

    while hand_value(player) < player_stand_at:
        player.append(deck.pop())
    if hand_value(player) > 21:
        return "dealer"

    while hand_value(dealer) < 17:
        dealer.append(deck.pop())
    if hand_value(dealer) > 21:
        return "player"

    if hand_value(player) > hand_value(dealer):
        return "player"
    if hand_value(player) < hand_value(dealer):
        return "dealer"
    return "draw"

for _ in range(5):
    print(play_game())

player
dealer
draw
player
player


## Simulating many games
One game tells us very little, because blackjack depends on luck. To compare strategies we play many games and compute the win rate:

$$\hat{p} = \frac{W}{N}$$

where $W$ is the number of games won by the player and $N$ is the total number of games. The more games we play, the closer $\hat{p}$ gets to the true probability of winning.

A strategy here is the value at which the player stops drawing. We compare stopping at 13, 15 and 17.

In [6]:
def win_rate(stand_at, n_games=10000):
    wins = 0
    for _ in range(n_games):
        if play_game(stand_at) == "player":
            wins += 1
    return wins / n_games
for threshold in [13, 15, 17]:
    print(threshold, win_rate(threshold))

13 0.42
15 0.4153
17 0.41


### Results
For each strategy we played 10,000 games. The player wins about 41-42% of them:

| Stop drawing at | Win rate |
|---|---|
| 13 | 0.42 |
| 15 | 0.4153 |
| 17 | 0.41 |

The three win rates are very close. Because the outcome depends on luck, every estimate has a random error of about $\pm 0.5\%$ with 10,000 games, so a difference of one percentage point is too small to name a clear best strategy.

The player wins less than half of the games. The main reason is that the player acts first: if the player goes over 21, the game is lost immediately, even in cases where the dealer would also have gone over 21.

To find out if one threshold is really better, we would need many more games per strategy.